In [72]:
import requests
import json
import time
from datetime import datetime, timedelta

api_key = "tao-60cae36a-63b0-4510-8af1-b47baa509447:15fac2d4" # API Key
start_date = "2024-01-01" # Startdatum
end_date = "2024-12-01" # Enddatum
coldkeys = ["5FL3X4Cscak28RJ2p3vyDe5CP9cgVroZgs7uXJ5ptCnWzyLD"] # Coldkey

price_url = "https://dash.taostats.io/api/v1/price/daily?start_date={}&end_date={}&api_key={}".format(start_date, end_date, api_key)
coldkey_url = "https://dash.taostats.io/api/v1/coldkey/daily?coldkey={}&start_date={}&end_date={}&api_key={}"
registration_url = "https://dash.taostats.io/api/v1/registration/daily?coldkey={}&start_date={}&end_date={}&api_key={}"

def fix_date(date_str):
    try:
        return datetime.strptime(date_str, "%Y-%m-%dT%H:%M:%S.%fZ").strftime("%Y-%m-%d")
    except ValueError:
        return datetime.strptime(date_str, "%Y-%m-%dT%H:%M:%SZ").strftime("%Y-%m-%d")

price_response = requests.get(price_url)
price_data = price_response.json()
prices = {item['date']: item['price'] for item in price_data['data']}

totals = {}
totals_csv = "coldkey,hotkey,date,income,expense,price\n"

for coldkey in coldkeys:
    coldkey_response = requests.get(coldkey_url.format(coldkey, start_date, end_date, api_key))
    coldkey_data = coldkey_response.json()
    if 'data' not in coldkey_data:
        print(f"No data for coldkey: {coldkey}")
        continue
    
    daily_data = coldkey_data['data']
    
    for i, day in enumerate(daily_data):
        date = fix_date(day['date'])
        hotkey = day['hotkey']
        income = 0
        expense = 0
        
        if i > 0:
            prev_day = daily_data[i - 1]
            income = day['balance'] - prev_day['balance'] if day['balance'] > prev_day['balance'] else 0
            expense = prev_day['balance'] - day['balance'] if day['balance'] < prev_day['balance'] else 0
            
        registration_response = requests.get(registration_url.format(coldkey, date, date, api_key))
        registration_data = registration_response.json()
        if 'data' in registration_data and len(registration_data['data']) > 0:
            for reg in registration_data['data']:
                if reg['hotkey'] == hotkey:
                    income += reg['amount']
                    
        prices_date = datetime.strptime(date, "%Y-%m-%d").strftime("%Y-%m-%dT00:00:00Z")
        price = prices.get(prices_date, 0)
        totals_csv += f"{coldkey},{hotkey},{date},{income},{expense},{price}\n"
        time.sleep(1) #Um ratelimit zu vermeiden
        
print(totals_csv)


In [73]:
import csv
import io

with open('incometotals.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    reader = csv.reader(io.StringIO(totals_csv), delimiter=',', quotechar='"')
    for row in reader:
        writer.writerow(row)